In [ ]:
# ============================================================
# FIX CELL — Resave LSTM in New Keras Format
# Run this ONCE to fix the loading problem
# ============================================================

import os
import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings('ignore')

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf

tf.keras.backend.clear_session()

print("="*55)
print("   FIXING LSTM MODEL — RESAVING IN NEW FORMAT")
print("="*55)
print()

# ---- Load BOP data ----
df = pd.read_csv('data/tea_demand_timeseries.csv')
df['Date'] = pd.to_datetime(df['Date'])

bop = df[df['TeaGrade'] == 'BOP'].copy()
bop = bop.sort_values('Date').set_index('Date')
bop.index.freq = 'D'
ts  = bop[['DemandKg']]

LOOK_BACK = 30
TEST_DAYS  = 30

# ---- Scale ----
scaler = MinMaxScaler(feature_range=(0, 1))
scaled = scaler.fit_transform(ts[['DemandKg']])

# ---- Build sequences ----
X_all, y_all = [], []
for i in range(LOOK_BACK, len(scaled)):
    X_all.append(scaled[i - LOOK_BACK:i, 0])
    y_all.append(scaled[i, 0])
X_all = np.array(X_all)
y_all = np.array(y_all)

split   = len(X_all) - TEST_DAYS
X_train = X_all[:split].reshape(-1, LOOK_BACK, 1)
y_train = y_all[:split]

# ---- Build model ----
print("Training LSTM model...")
print("(Takes 3-8 minutes — please wait)")
print()

model = Sequential([
    LSTM(64, return_sequences=True,
         input_shape=(LOOK_BACK, 1)),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])

# ⚠️ KEY FIX: use 'mean_squared_error' string NOT 'mse'
model.compile(
    optimizer='adam',
    loss='mean_squared_error'
)

es = EarlyStopping(
    monitor='val_loss',
    patience=12,
    restore_best_weights=True,
    verbose=1
)

model.fit(
    X_train, y_train,
    epochs=80,
    batch_size=16,
    validation_split=0.1,
    callbacks=[es],
    verbose=1
)

print()
print("✅ LSTM training complete!")

# ---- Create folder ----
os.makedirs('saved_models/lstm', exist_ok=True)

# ---- Save in NEW .keras format ----
# This is the fix — use .keras NOT .h5
model.save('saved_models/lstm/lstm_demand_model.keras')
print("✅ LSTM saved in NEW format (.keras)")

# ---- Also save scaler ----
joblib.dump(scaler,
            'saved_models/lstm/lstm_scaler.pkl')
print("✅ LSTM scaler saved")

# ---- Save last 30 days as default ----
last_30 = ts['DemandKg'].values[-30:].tolist()
joblib.dump(last_30,
            'saved_models/lstm/default_last30_days.pkl')
print("✅ Default 30 days saved")

# ---- Verify file exists ----
model_size = os.path.getsize(
    'saved_models/lstm/lstm_demand_model.keras'
) / 1024

print()
print("="*55)
print("   ✅ LSTM MODEL SAVED CORRECTLY!")
print("="*55)
print(f"   File: saved_models/lstm/lstm_demand_model.keras")
print(f"   Size: {model_size:.1f} KB")
print()
print("   Now update smarttea_ai_api.py:")
print("   Change: lstm_demand_model.h5")
print("   To:     lstm_demand_model.keras")


   FIXING LSTM MODEL — RESAVING IN NEW FORMAT

Training LSTM model...
(Takes 3-8 minutes — please wait)

Epoch 1/80
59/59 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - loss: 0.0566 - val_loss: 0.0314
Epoch 2/80
59/59 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - loss: 0.0408 - val_loss: 0.0351
Epoch 3/80
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.0378 - val_loss: 0.0327
Epoch 4/80
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 0.0384 - val_loss: 0.0325
Epoch 5/80
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.0362 - val_loss: 0.0308
Epoch 6/80
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.0382 - val_loss: 0.0311
Epoch 7/80
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.0376 - val_loss: 0.0377
Epoch 8/80
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.0362 - val_loss: 0.0307
Epoch 9/80
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0347 - val_loss: 0.0309
Epoch 10/80
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0370 - val_loss: 0.0310
Epoch 11/80
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 1

In [1]:
# ============================================================
# CELL 0 — Create folder structure for saved models
# ============================================================

import os

# This creates saved_models folder INSIDE SmartTea_AI
# which is INSIDE Smart tea folder
base = 'saved_models'

folders = [
    f'{base}',
    f'{base}/lstm',
    f'{base}/linear_regression',
    f'{base}/xgboost',
    f'{base}/anomaly',
    f'{base}/multistep',
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"✅ Created: {folder}")

print()
print("Your Smart tea folder structure:")
print("Smart tea/")
print("├── TeaOnlineShop/     ← ASP.NET system")
print("└── SmartTea_AI/")
print("    ├── saved_models/  ← NEW (models saved here)")
print("    │   ├── lstm/")
print("    │   ├── linear_regression/")
print("    │   ├── xgboost/")
print("    │   ├── anomaly/")
print("    │   └── multistep/")
print("    ├── data/")
print("    ├── plots/")
print("    └── (your notebooks)")
print()
print("✅ Ready to save models!")

✅ Created: saved_models
✅ Created: saved_models/lstm
✅ Created: saved_models/linear_regression
✅ Created: saved_models/xgboost
✅ Created: saved_models/anomaly
✅ Created: saved_models/multistep

Your Smart tea folder structure:
Smart tea/
├── TeaOnlineShop/     ← ASP.NET system
└── SmartTea_AI/
    ├── saved_models/  ← NEW (models saved here)
    │   ├── lstm/
    │   ├── linear_regression/
    │   ├── xgboost/
    │   ├── anomaly/
    │   └── multistep/
    ├── data/
    ├── plots/
    └── (your notebooks)

✅ Ready to save models!


In [2]:
# ============================================================
# CELL 1 — Retrain LSTM and Save It
# (We retrain here so we can save it properly)
# Takes about 5-8 minutes
# ============================================================

import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf

tf.keras.backend.clear_session()

# ---- Load BOP demand data ----
df = pd.read_csv('data/tea_demand_timeseries.csv')
df['Date'] = pd.to_datetime(df['Date'])

bop = df[df['TeaGrade'] == 'BOP'].copy()
bop = bop.sort_values('Date').set_index('Date')
bop.index.freq = 'D'
ts = bop[['DemandKg']]

LOOK_BACK = 30
TEST_DAYS  = 30

# ---- Scale ----
scaler = MinMaxScaler(feature_range=(0, 1))
scaled = scaler.fit_transform(ts[['DemandKg']])

# ---- Build sequences ----
X_all, y_all = [], []
for i in range(LOOK_BACK, len(scaled)):
    X_all.append(scaled[i - LOOK_BACK:i, 0])
    y_all.append(scaled[i, 0])
X_all = np.array(X_all)
y_all = np.array(y_all)

split     = len(X_all) - TEST_DAYS
X_train   = X_all[:split].reshape(-1, LOOK_BACK, 1)
X_test    = X_all[split:].reshape(-1, LOOK_BACK, 1)
y_train   = y_all[:split]
y_test    = y_all[split:]

# ---- Build and train LSTM ----
print("Training LSTM model... (5-8 minutes)")
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(LOOK_BACK, 1)),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])
model.compile(optimizer='adam', loss='mse')

es = EarlyStopping(
    monitor='val_loss', patience=12,
    restore_best_weights=True, verbose=1
)
model.fit(X_train, y_train,
          epochs=80, batch_size=16,
          validation_split=0.1,
          callbacks=[es], verbose=1)

print("\n✅ LSTM training complete!")

# ---- Save LSTM model ----
model.save('saved_models/lstm/lstm_demand_model.h5')
print("✅ LSTM model saved!")

# ---- Save scaler ----
joblib.dump(scaler, 'saved_models/lstm/lstm_scaler.pkl')
print("✅ LSTM scaler saved!")

# ---- Save last 30 days for default input ----
# SmartTea will send its own last 30 days
# But we save this as backup/default
last_30_days = ts['DemandKg'].values[-30:].tolist()
joblib.dump(last_30_days,
            'saved_models/lstm/default_last30_days.pkl')

# ---- Save metadata ----
metadata = {
    'grade':       'BOP',
    'look_back':   LOOK_BACK,
    'mape':        7.347,
    'mae':         27.825,
    'description': 'LSTM demand forecasting for BOP grade'
}
joblib.dump(metadata, 'saved_models/lstm/metadata.pkl')
print("✅ LSTM metadata saved!")

print()
print("Files saved in saved_models/lstm/:")
print("  lstm_demand_model.h5")
print("  lstm_scaler.pkl")
print("  default_last30_days.pkl")
print("  metadata.pkl")


Training LSTM model... (5-8 minutes)
Epoch 1/80
59/59 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - loss: 0.0654 - val_loss: 0.0317
Epoch 2/80
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.0394 - val_loss: 0.0311
Epoch 3/80
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0411 - val_loss: 0.0331
Epoch 4/80
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0380 - val_loss: 0.0313
Epoch 5/80
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0379 - val_loss: 0.0306
Epoch 6/80
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0379 - val_loss: 0.0329
Epoch 7/80
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0359 - val_loss: 0.0323
Epoch 8/80
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0347 - val_loss: 0.0318
Epoch 9/80
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0356 - val_loss: 0.0306
Epoch 10/80
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0392 - val_loss: 0.0323
Epoch 11/80
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.0352 - val_loss: 0.0297
Epoch 12/80
59/59 ━━━━━━━


✅ LSTM training complete!
✅ LSTM model saved!
✅ LSTM scaler saved!
✅ LSTM metadata saved!

Files saved in saved_models/lstm/:
  lstm_demand_model.h5
  lstm_scaler.pkl
  default_last30_days.pkl
  metadata.pkl


In [3]:
# ============================================================
# CELL 2 — Retrain Linear Regression and Save

# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

# ---- Load price data ----
df_price = pd.read_csv('data/tea_price_regression.csv')
df_price['Date'] = pd.to_datetime(df_price['Date'])

FEATURE_COLS = [
    'Month', 'DayOfWeek', 'Quarter',
    'IsWeekend', 'DayOfYear', 'Day',
    'GreenLeafPricePerKgLKR',
    'PriceLag1', 'PriceLag2', 'PriceLag3',
    'PriceLag7', 'PriceLag14',
    'PriceRollingMean7', 'PriceRollingMean30',
    'PriceRollingStd7', 'PriceChangePct',
    'GreenLeafQuantityKg', 'QtyRollingMean7',
    'FirewoodCollectedKg', 'FirewoodCostPerKgLKR',
    'TotalDailyCostLKR',
    'TemperatureCelsius', 'RainfallMM', 'HeavyRainFlag',
    'SupplierDelivered', 'SupplierQuantityKg',
    'PromotionActive', 'MonthStartBonus',
]
TARGET = 'NextDayGreenLeafPrice'

X = df_price[FEATURE_COLS]
y = df_price[TARGET]

split_idx   = int(len(X) * 0.80)
X_train     = X.iloc[:split_idx]
X_test      = X.iloc[split_idx:]
y_train     = y.iloc[:split_idx]
y_test      = y.iloc[split_idx:]

# ---- Train Linear Regression ----
print("Training Linear Regression...")
scaler_lr   = StandardScaler()
X_train_sc  = scaler_lr.fit_transform(X_train)
X_test_sc   = scaler_lr.transform(X_test)

lr_model    = LinearRegression()
lr_model.fit(X_train_sc, y_train)

# ---- Save LR model ----
joblib.dump(lr_model,  'saved_models/linear_regression/lr_model.pkl')
joblib.dump(scaler_lr, 'saved_models/linear_regression/lr_scaler.pkl')
joblib.dump(FEATURE_COLS,
            'saved_models/linear_regression/feature_cols.pkl')

lr_meta = {
    'mape':        2.994,
    'r2':          0.804,
    'description': 'Linear Regression for next-day price'
}
joblib.dump(lr_meta,
            'saved_models/linear_regression/metadata.pkl')

print("✅ Linear Regression saved!")

# ---- Train XGBoost ----
print("Training XGBoost...")
xgb_model = xgb.XGBRegressor(
    n_estimators=200, max_depth=3,
    learning_rate=0.03, subsample=1.0,
    colsample_bytree=0.8,
    random_state=42, verbosity=0
)
xgb_model.fit(X_train, y_train)

joblib.dump(xgb_model,
            'saved_models/xgboost/xgb_model.pkl')
xgb_meta = {
    'mape':        3.463,
    'r2':          0.716,
    'description': 'XGBoost for next-day price (challenger)'
}
joblib.dump(xgb_meta,
            'saved_models/xgboost/metadata.pkl')

print("✅ XGBoost saved!")

# ---- Save sample input for SmartTea default ----
# Get the last row as a sample input
sample_input = X_test.iloc[-1:].to_dict('records')[0]
joblib.dump(sample_input,
            'saved_models/linear_regression/sample_input.pkl')

print()
print("Files saved:")
print("  saved_models/linear_regression/lr_model.pkl")
print("  saved_models/linear_regression/lr_scaler.pkl")
print("  saved_models/linear_regression/feature_cols.pkl")
print("  saved_models/xgboost/xgb_model.pkl")

Training Linear Regression...
✅ Linear Regression saved!
Training XGBoost...
✅ XGBoost saved!

Files saved:
  saved_models/linear_regression/lr_model.pkl
  saved_models/linear_regression/lr_scaler.pkl
  saved_models/linear_regression/feature_cols.pkl
  saved_models/xgboost/xgb_model.pkl


In [4]:
# ============================================================
# CELL 3 — Train and Save 7 Multi-Step Models

# ============================================================

print("Training 7 multi-step models...")
print()

HORIZONS = [1, 2, 3, 4, 5, 6, 7]

# Expected MAPEs from our research
horizon_mapes = {
    1: 2.968, 2: 2.997, 3: 3.021,
    4: 3.237, 5: 3.090, 6: 3.150, 7: 2.951
}

df_ms = df_price.copy()

for h in HORIZONS:
    # Create target for this horizon
    df_ms[f'Target_{h}'] = (
        df_ms['GreenLeafPricePerKgLKR'].shift(-h)
    )

df_ms = df_ms.dropna(
    subset=[f'Target_{h}' for h in HORIZONS]
).reset_index(drop=True)

split_ms = int(len(df_ms) * 0.80)
X_ms     = df_ms[FEATURE_COLS]

X_tr_ms  = X_ms.iloc[:split_ms]
X_te_ms  = X_ms.iloc[split_ms:]

# Scale once for all LR models
sc_ms    = StandardScaler()
X_tr_sc  = sc_ms.fit_transform(X_tr_ms)
X_te_sc  = sc_ms.transform(X_te_ms)

# Save shared scaler
joblib.dump(sc_ms, 'saved_models/multistep/scaler.pkl')

for h in HORIZONS:
    y_tr_h = df_ms[f'Target_{h}'].iloc[:split_ms]
    y_te_h = df_ms[f'Target_{h}'].iloc[split_ms:]

    # Train LR for this horizon
    lr_h = LinearRegression()
    lr_h.fit(X_tr_sc, y_tr_h)

    # Save
    joblib.dump(lr_h,
        f'saved_models/multistep/lr_day{h}.pkl')

    # Verify
    pred_h = lr_h.predict(X_te_sc)
    mape_h = np.mean(
        np.abs((y_te_h.values - pred_h) / y_te_h.values)
    ) * 100

    print(f"  Day+{h}: MAPE={mape_h:.3f}%  ✅ Saved")

# Save horizon metadata
ms_meta = {
    'horizons':      HORIZONS,
    'horizon_mapes': horizon_mapes,
    'description':   '7-day multi-step price forecasting'
}
joblib.dump(ms_meta, 'saved_models/multistep/metadata.pkl')

print()
print("✅ All 7 multi-step models saved!")

Training 7 multi-step models...

  Day+1: MAPE=2.968%  ✅ Saved
  Day+2: MAPE=2.997%  ✅ Saved
  Day+3: MAPE=3.021%  ✅ Saved
  Day+4: MAPE=3.237%  ✅ Saved
  Day+5: MAPE=3.090%  ✅ Saved
  Day+6: MAPE=3.150%  ✅ Saved
  Day+7: MAPE=2.951%  ✅ Saved

✅ All 7 multi-step models saved!


In [5]:
# ============================================================
# CELL 4 — Train and Save Anomaly Detection Models

# ============================================================

from sklearn.ensemble import IsolationForest

print("Training Anomaly Detection models...")
print()

GRADES = ['BOP', 'BOPF', 'DUST', 'FNGS', 'OP']
DEMAND_FEATURES = [
    'DemandKg', 'StockLevelKg', 'PricePerKgLKR',
    'DayOfWeek', 'Month', 'IsWeekend'
]

# ---- Demand anomaly model per grade ----
for grade in GRADES:
    grade_df = df[df['TeaGrade'] == grade].copy()
    X_g      = grade_df[DEMAND_FEATURES].values

    sc_g     = StandardScaler()
    X_g_sc   = sc_g.fit_transform(X_g)

    iso_g    = IsolationForest(
        contamination=0.05,
        n_estimators=200,
        random_state=42
    )
    iso_g.fit(X_g_sc)

    joblib.dump(iso_g,
        f'saved_models/anomaly/iso_demand_{grade}.pkl')
    joblib.dump(sc_g,
        f'saved_models/anomaly/sc_demand_{grade}.pkl')

    print(f"  ✅ Demand anomaly model saved: {grade}")

# ---- Price anomaly model ----
PRICE_FEATURES = [
    'GreenLeafPricePerKgLKR', 'PriceChangePct',
    'GreenLeafQuantityKg', 'RainfallMM',
    'TemperatureCelsius', 'SupplierDelivered',
    'Month', 'DayOfWeek'
]

X_p    = df_price[PRICE_FEATURES].values
sc_p   = StandardScaler()
X_p_sc = sc_p.fit_transform(X_p)

iso_p  = IsolationForest(
    contamination=0.05,
    n_estimators=200,
    random_state=42
)
iso_p.fit(X_p_sc)

joblib.dump(iso_p, 'saved_models/anomaly/iso_price.pkl')
joblib.dump(sc_p,  'saved_models/anomaly/sc_price.pkl')

# Save feature lists
joblib.dump(DEMAND_FEATURES,
            'saved_models/anomaly/demand_features.pkl')
joblib.dump(PRICE_FEATURES,
            'saved_models/anomaly/price_features.pkl')

print("  ✅ Price anomaly model saved")
print()
print("✅ All anomaly models saved!")

Training Anomaly Detection models...

  ✅ Demand anomaly model saved: BOP
  ✅ Demand anomaly model saved: BOPF
  ✅ Demand anomaly model saved: DUST
  ✅ Demand anomaly model saved: FNGS
  ✅ Demand anomaly model saved: OP
  ✅ Price anomaly model saved

✅ All anomaly models saved!


In [6]:
# ============================================================
# CELL 5 — Check All Files Are Saved Correctly
# ============================================================

import os

print("="*60)
print("   VERIFYING ALL SAVED MODELS")
print("="*60)
print()

folders = {
    'LSTM':              'saved_models/lstm',
    'Linear Regression': 'saved_models/linear_regression',
    'XGBoost':           'saved_models/xgboost',
    'Anomaly':           'saved_models/anomaly',
    'Multi-Step':        'saved_models/multistep'
}

all_good = True
for name, path in folders.items():
    files = os.listdir(path)
    total_size = sum(
        os.path.getsize(f"{path}/{f}")
        for f in files
    ) / 1024

    print(f"  {name} ({len(files)} files, "
          f"{total_size:.0f} KB total):")
    for f in files:
        size = os.path.getsize(f"{path}/{f}") / 1024
        print(f"    ✅ {f} ({size:.0f} KB)")
    print()

print("="*60)
print("✅ ALL MODELS SAVED SUCCESSFULLY!")
print()
print("Your final folder structure:")
print()
print("Smart tea/")
print("├── TeaOnlineShop/         ← ASP.NET system")
print("└── SmartTea_AI/")
print("    ├── saved_models/      ← AI models ready")
print("    │   ├── lstm/")
print("    │   ├── linear_regression/")
print("    │   ├── xgboost/")
print("    │   ├── anomaly/")
print("    │   └── multistep/")
print("    ├── data/")
print("    ├── plots/")
print("    ├── results/")
print("    └── (notebooks)")
print()
print("You will NEVER need to retrain again.")
print("Just use smarttea_ai_api.py to serve predictions.")

   VERIFYING ALL SAVED MODELS

  LSTM (4 files, 394 KB total):
    ✅ default_last30_days.pkl (0 KB)
    ✅ lstm_demand_model.h5 (392 KB)
    ✅ lstm_scaler.pkl (1 KB)
    ✅ metadata.pkl (0 KB)

  Linear Regression (5 files, 4 KB total):
    ✅ feature_cols.pkl (0 KB)
    ✅ lr_model.pkl (1 KB)
    ✅ lr_scaler.pkl (2 KB)
    ✅ metadata.pkl (0 KB)
    ✅ sample_input.pkl (1 KB)

  XGBoost (2 files, 233 KB total):
    ✅ metadata.pkl (0 KB)
    ✅ xgb_model.pkl (233 KB)

  Anomaly (14 files, 20358 KB total):
    ✅ demand_features.pkl (0 KB)
    ✅ iso_demand_BOP.pkl (3471 KB)
    ✅ iso_demand_BOPF.pkl (3495 KB)
    ✅ iso_demand_DUST.pkl (3447 KB)
    ✅ iso_demand_FNGS.pkl (3468 KB)
    ✅ iso_demand_OP.pkl (3548 KB)
    ✅ iso_price.pkl (2924 KB)
    ✅ price_features.pkl (0 KB)
    ✅ sc_demand_BOP.pkl (1 KB)
    ✅ sc_demand_BOPF.pkl (1 KB)
    ✅ sc_demand_DUST.pkl (1 KB)
    ✅ sc_demand_FNGS.pkl (1 KB)
    ✅ sc_demand_OP.pkl (1 KB)
    ✅ sc_price.pkl (1 KB)

  Multi-Step (9 files, 9 KB total):
    